#### Import libraries & set seeds

In [ ]:
# Standard Libaries
import sys
from pathlib import Path

# Setting Path
notebook_path = Path().resolve()
parent_dir = notebook_path.parent
src_dir = parent_dir / "src"
sys.path.append(str(parent_dir))
sys.path.append(str(src_dir))

import math
import numpy as np
import torch
from src.dataset import load_image_dataframe, ImageDataset
from src.transforms import get_transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import random
import pytorch_lightning as pl


# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pl.seed_everything(seed, workers=True, verbose=False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(15)

#### Load data

In [ ]:
full_dataset = load_image_dataframe(data_path=parent_dir / "data/iqa", split="train", exclusion=False, binary=False)
dataset = ImageDataset(full_dataset, transform=get_transforms(train=False, normalize=False), preprocess=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

#### Visualize samples

In [ ]:
def cache_images_for_all_oiq(loader):
    oiq_cache = {}
    needed_oiq = {0, 1, 2} - oiq_cache.keys()  # only cache what’s missing
    if not needed_oiq:
        return  # all values already cached

    # Initialize empty dicts
    for o in needed_oiq:
        oiq_cache[o] = {(m, e): None for m in range(3) for e in range(3)}

    for batch in loader:
        images = batch[0]
        mucosal, expansion, oiq = batch[1], batch[2], batch[3]

        for i in range(images.size(0)):
            o = oiq[i].item()
            m = mucosal[i].item()
            e = expansion[i].item()

            if o not in needed_oiq:
                continue
            if -100 in (m, e) or m not in (0, 1, 2) or e not in (0, 1, 2):
                continue

            key = (m, e)
            if oiq_cache[o][key] is None:
                oiq_cache[o][key] = images[i].cpu().clone()

        # Check early stopping condition
        for o in list(needed_oiq):
            if all(oiq_cache[o][k] is not None for k in oiq_cache[o]):
                needed_oiq.remove(o)

        if not needed_oiq:
            break  # done caching all

    # Final check
    for o in (0, 1, 2):
        assert o in oiq_cache, f"OIQ {o} not cached"
        assert all(oiq_cache[o][k] is not None for k in oiq_cache[o]), f"Incomplete cache for OIQ {o}"

    return oiq_cache


oiq_cache = cache_images_for_all_oiq(loader)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
import torch

# Mapping from index to label
label_map = {0: "Poor", 1: "Adequate", 2: "Good"}


def draw_figure_bbox_around_axes(fig, axes, color="red", linewidth=6):
    """Draw a bounding box around a group of axes in figure coordinates."""
    bboxes = [ax.get_position() for ax in axes]

    x0 = min(b.x0 for b in bboxes)
    y0 = min(b.y0 for b in bboxes)
    x1 = max(b.x1 for b in bboxes)
    y1 = max(b.y1 for b in bboxes)

    width = x1 - x0
    height = y1 - y0

    rect = Rectangle((x0, y0), width, height, transform=fig.transFigure, fill=False, color=color, linewidth=linewidth)
    fig.patches.append(rect)

    return (x0, x1, y0, y1)


def add_oiq_label_top(fig, axes, text="OIQ = Poor", color="red", fontsize=24):
    """Place OIQ label centered above the grid."""
    bboxes = [ax.get_position() for ax in axes]
    x0 = min(b.x0 for b in bboxes)
    x1 = max(b.x1 for b in bboxes)
    y1 = max(b.y1 for b in bboxes)
    x_center = (x0 + x1) / 2

    fig.text(x_center, y1 + 0.06, text, ha="center", va="bottom", fontsize=fontsize, color=color, fontweight="bold")


def add_axis_labels(fig, axes_grid, label_map, fontsize=20):
    """Add x-category labels below and y-category labels beside."""
    ref_axes = axes_grid[-1]  # bottom row

    # Column labels BELOW bottom row
    for col_idx, ax in enumerate(ref_axes):
        bbox = ax.get_position()
        x_center = (bbox.x0 + bbox.x1) / 2
        y_bottom = bbox.y0
        fig.text(x_center, y_bottom - 0.02, label_map[col_idx], ha="center", va="top", fontsize=fontsize)

    # Row labels (rotated on the left of each row)
    for row_idx, row_axes in enumerate(axes_grid):
        bbox = row_axes[0].get_position()
        x_left = bbox.x0
        y_center = (bbox.y0 + bbox.y1) / 2
        fig.text(x_left - 0.03, y_center, label_map[row_idx], ha="center", va="center", rotation=90, fontsize=fontsize)


def add_overarching_axis_labels(fig, axes_grid, x_label="Expansion", y_label="Mucosal Cleaning", fontsize=20):
    # Horizontal (x) axis label centered under all columns
    bottom_row = axes_grid[-1]
    bbox0 = bottom_row[0].get_position()
    bboxN = bottom_row[-1].get_position()
    x_center = (bbox0.x0 + bboxN.x1) / 2
    y_bottom = min(bbox0.y0, bboxN.y0)

    fig.text(
        x_center,
        y_bottom - 0.06,
        x_label,
        ha="center",
        va="top",
        fontsize=fontsize,
        fontweight="bold",
    )

    # Vertical (y) axis label centered beside all rows
    first_col = [row[0] for row in axes_grid]
    bbox0 = first_col[0].get_position()
    bboxN = first_col[-1].get_position()
    y_center = (bbox0.y1 + bboxN.y0) / 2
    x_left = bbox0.x0

    fig.text(
        x_left - 0.08,
        y_center,
        y_label,
        ha="center",
        va="center",
        rotation=90,
        fontsize=fontsize,
        fontweight="bold",
    )


def visualize_cached_oiq_panel(oiq_cache, oiq_value, denormalize_fn=None):
    assert oiq_value in oiq_cache, f"OIQ {oiq_value} not cached yet"
    if oiq_value == 0:
        color = "red"
    elif oiq_value == 1:
        color = "orange"
    elif oiq_value == 2:
        color = "green"

    selected = oiq_cache[oiq_value]

    fig = plt.figure(figsize=(9, 9), dpi=256)
    gs = GridSpec(nrows=3, ncols=3, figure=fig, wspace=0.01, hspace=0.01)

    axes_grid = []

    for m in range(3):
        row_axes = []
        for e in range(3):
            ax = fig.add_subplot(gs[m, e])
            ax.set_aspect("equal")
            ax.axis("off")
            row_axes.append(ax)

            img_tensor = selected[(m, e)]
            if img_tensor is not None:
                img = denormalize_fn(img_tensor) if denormalize_fn else img_tensor
                img = torch.clamp(img.cpu(), 0, 1).permute(1, 2, 0).numpy()
                ax.imshow(img)
        axes_grid.append(row_axes)

    # Flatten axes for layout reference
    all_axes = [ax for row in axes_grid for ax in row]

    # Draw bounding box and add top label
    draw_figure_bbox_around_axes(fig, all_axes, color=color)
    add_oiq_label_top(fig, all_axes, text=f"OIQ = {label_map[oiq_value]}", color=color)

    # Add category and axis labels
    add_axis_labels(fig, axes_grid, label_map)
    add_overarching_axis_labels(fig, axes_grid, x_label="Expansion", y_label="Mucosal Cleaning")

    plt.show()
    return fig

In [ ]:
oiq_value = 0
fig = visualize_cached_oiq_panel(oiq_cache, oiq_value=oiq_value)
# fig.savefig(notebook_path / f"oiq_grid_visualization_{oiq_value}.png", bbox_inches="tight")